In [50]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [51]:
def relu(x):
    return torch.maximum(x, torch.tensor(0.0))

def relu_derivative(x):
    return (x > 0).float()

def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

In [52]:
def generate_data(n_samples=200 , n_features = 3 , noise = 0.2):
    x = torch.rand(n_samples , n_features)
    true_weights = torch.randn(n_features , 1)
    y = x @ true_weights + torch.randn(n_samples , 1) * noise
    return x,y

In [53]:
x , y_true = generate_data(n_samples=300 , n_features= 3 , noise=0.15)
print(f"x shape {x.shape}")
print(f"y_true shape {y_true.shape}")

x shape torch.Size([300, 3])
y_true shape torch.Size([300, 1])


In [54]:
def normalize(x):
    min_val = x.min(dim = 0)[0]
    max_val = x.max(dim = 0)[0]
    return ( x - min_val) / (max_val - min_val + 1e-8)

x = normalize(x)

In [55]:
def train_test_split_tensor(x , y , test_size = 0.2):
    n_samples = x.shape[0]
    n_test = int(n_samples * test_size)
    indices = torch.randperm(n_samples)
    test_indices = indices[:n_test]
    train_indices = indices[n_test:]
    x_train = x[train_indices]
    y_train = y[train_indices]
    x_test = x[test_indices]
    y_test = y[test_indices]
    return x_train, x_test , y_train , y_test

x_train, x_test, y_train, y_test = train_test_split_tensor(x, y_true, test_size=0.2)

In [56]:
class NeuralNetwork:
    def __init__(self , input_size , hidden_size , output_size , task = 'regression' ):
        self.w1 = torch.randn(input_size , hidden_size) * 0.01
        self.b1 = torch.randn(hidden_size) * 0.01
        self.w2 = torch.randn(hidden_size , output_size)* 0.01
        self.b2 = torch.randn(output_size) * 0.01
        self.task = task # regression / classification

        self.w1.requires_grad = False
        self.b1.requires_grad = False
        self.w2.requires_grad = False
        self.b2.requires_grad = False

        self.z1 = None
        self.a1 = None


    def forward(self, X):
      #hidden layer 
      # z1 = x @ w1 + b1
      # x : (n_samples , input_size)
      # w1 : (input_size , hidden_size)
      # x @ w1 : (n_samples , hidden_size)
      # + b1 : (hiddden_size,)

      self.z1 = X @ self.w1 + self.b1
      
      # a1 = RELU(z1)
      #اعداد منفی را صفر میکنیم و اعداد مثبت را نگه میداریم 
      self.a1 = relu(self.z1)


      # output layer 
      # z2 = a1 @ w2 + b2
      # a1 : (n_samples , hidden_size)
      # w2: (hidden_size , output_size)
      # a1 @ w2 : (n_samples , output_size)
      # + b2 : (output_size,)
      z2 = self.a1 @ self.w2 + self.b2

      if self.task == 'regression':
             # y_pred = z2 (regression)
              y_pred = z2
      elif self.task == 'classification':
             y_pred = sigmoid(z2)
      else :
          raise ValueError("error please select regression or classification")
      return y_pred

In [57]:
def mse_loss(y_pred , y_true):
    return ((y_pred - y_true) ** 2).mean()

def mse_derivative(y_pred , y_true): # derivation of mse with respect to y_pred
    # formola : (2/n)* (y_pred - y_true)
    return 2 * (y_pred - y_true ) / y_pred.size(0)

In [58]:
def train_step(model , x , y_true , learning_rate = 0.01):
    #train = forward + backward + update

    #داده ها را به شبکه میدیم 
    y_pred = model.forward(x)

    #loss
    loss = mse_loss(y_pred , y_true)

    #zero gradگرادیان های قبلی را صفر میکنیم 
    grad_w1 = torch.zeros_like(model.w1)
    grad_b1 = torch.zeros_like(model.b1)
    grad_w2 = torch.zeros_like(model.w2)
    grad_b2 = torch.zeros_like(model.b2)

    #backward propagation
    #مشتق خطا نسبت به خروجی
    dL_dy = mse_derivative(y_pred , y_true)

    #برای لایه دوم چون خروجی رگرسیون و خطی پس مشتقمون برابر یکه
    dL_dz2 = dL_dy # (n_sample , 1)
    # !!!!!!!!!!!!!!   dL/dW2 = dL/dz2 * dz2/dW2 = dL/dz2 * a1  ولی الان ضرب ماتریسی درست نیست چون سطرو ستوناشون با هم نمیخونه 
    # پس یاید ترنسپوز کنیم dL/dW2 = a1^T @ dL/dz2
    # w2: dL/dw2 = a1^T @ dl/dz2
     # a1: (n_samples, hidden_size) → a1^T: (hidden_size, n_samples)
    # dL_dz2: (n_samples, 1)
    # نتیجه: (hidden_size, 1)
    grad_w2 += model.a1.T @ dL_dz2
    grad_b2 += dL_dz2.sum(axis = 0)
    # a1 از رلو عبور میکرد 
    # dL/da1 = dL/dz2 @ W2^T
    # dL_dz2: (n_samples, 1) → dL_dz2 @ W2^T: (n_samples, hidden_size)
    dL_da1 = dL_dz2 @ model.w2.T
    #مشتق رلو برای زد یک های مثبت  یکه و برای  زد های منفی صقره
    dL_dz1 = dL_da1 * relu_derivative(model.z1)
    # مشتق نسبت به W1: dL/dW1 = X^T @ dL/dz1
    # X: (n_samples, input_size) → X^T: (input_size, n_samples)
    # dL_dz1: (n_samples, hidden_size)
    # نتیجه: (input_size, hidden_size)
    grad_w1 += x.T @ dL_dz1
    # مشتق نسبت به b1: dL/db1 = sum(dL/dz1) روی بعد نمونه‌ها
    # نتیجه: (hidden_size,)
    grad_b1 += dL_dz1.sum(axis=0)

    '''
    1. مشتق خطا نسبت به خروجی:
   dL/dy_pred = (2/n) * (y_pred - y_true)

   2. لایه خروجی (z2 = a1 @ W2 + b2):
   dL/dz2 = dL/dy_pred * 1  (چون مشتق خطی = 1)
   dL/dW2 = a1^T @ dL/dz2
   dL/db2 = sum(dL/dz2)

   3. لایه پنهان (a1 = ReLU(z1)):
   dL/da1 = dL/dz2 @ W2^T
   dL/dz1 = dL/da1 * ReLU'(z1)
   dL/dW1 = X^T @ dL/dz1
   dL/db1 = sum(dL/dz1)
    '''

    #weights update
    # w_new = w_old - learning_rate * dl/dw
    model.w1 -= learning_rate * grad_w1
    model.b1 -= learning_rate * grad_b1
    model.w2 -= learning_rate * grad_w2
    model.b2 -= learning_rate * grad_b2



    return loss.item()

In [61]:

input_size = x.shape[1]      # تعداد ویژگی‌ها 
hidden_size = 10             
output_size = 1              
learning_rate = 0.01         
epochs = 200                 


model = NeuralNetwork(input_size, hidden_size, output_size, task = 'regression')

print(f"W1 shape: {model.w1.shape}")  
print(f"b1 shape: {model.b1.shape}")  
print(f"W2 shape: {model.w2.shape}")  
print(f"b2 shape: {model.b2.shape}") 


loss_history = []  


for epoch in range(epochs):
    # یک مرحله آموزش
    loss = train_step(model, x_train, y_train, learning_rate)
    loss_history.append(loss)
    
    # هر 20 دوره، خطا را چاپ کن
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}/{epochs} - Loss: {loss:.6f}")


W1 shape: torch.Size([3, 10])
b1 shape: torch.Size([10])
W2 shape: torch.Size([10, 1])
b2 shape: torch.Size([1])
Epoch   0/200 - Loss: 0.719814
Epoch  20/200 - Loss: 0.463206
Epoch  40/200 - Loss: 0.348914
Epoch  60/200 - Loss: 0.298012
Epoch  80/200 - Loss: 0.275345
Epoch 100/200 - Loss: 0.265251
Epoch 120/200 - Loss: 0.260753
Epoch 140/200 - Loss: 0.258747
Epoch 160/200 - Loss: 0.257848
Epoch 180/200 - Loss: 0.257443
